# Route + Layovers: How Much Does Price Change?

For each of our 20 routes, show average price at each stop count (0, 1, 2, 3), and the exact dollar change from nonstop to each layover tier. Sample size (n) is shown so unreliable numbers (built from very few rows) are obvious.

In [1]:
import polars as pl

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_width_chars(200)

df = pl.read_csv("../flight_sample.csv")

df = df.with_columns(
    (pl.col("segmentsAirlineCode").str.split("||").list.len() - 1).alias("numStops")
)
df = df.with_columns(
    (pl.col("startingAirport") + "-" + pl.col("destinationAirport")).alias("route")
)

route_stops = (
    df.group_by(["route", "numStops"])
    .agg(pl.col("totalFare").mean().alias("avgFare"), pl.len().alias("n"))
    .sort(["route", "numStops"])
)
route_stops.head(10)

route,numStops,avgFare,n
str,u32,f64,u32
"""ATL-LAX""",0,406.512017,18062
"""ATL-LAX""",1,356.712119,67868
"""ATL-LAX""",2,413.613856,1058
"""BOS-LAX""",0,439.773329,20165
"""BOS-LAX""",1,379.099464,57562
"""BOS-LAX""",2,472.597851,1047
"""CLT-LAX""",0,432.921673,7399
"""CLT-LAX""",1,422.976737,58672
"""CLT-LAX""",2,528.136113,3738


## Pivoted: one row per route, price at each stop count, and dollar change between tiers

In [2]:
wide = route_stops.pivot(on="numStops", index="route", values=["avgFare", "n"])

wide = wide.rename({
    "avgFare_0": "fare_0stop", "n_0": "n_0stop",
    "avgFare_1": "fare_1stop", "n_1": "n_1stop",
    "avgFare_2": "fare_2stop", "n_2": "n_2stop",
    "avgFare_3": "fare_3stop", "n_3": "n_3stop",
})

wide = wide.with_columns([
    (pl.col("fare_1stop") - pl.col("fare_0stop")).round(2).alias("drop_0to1"),
    (pl.col("fare_2stop") - pl.col("fare_1stop")).round(2).alias("drop_1to2"),
])

wide = wide.sort("drop_0to1")

wide.select(["route", "fare_0stop", "n_0stop", "fare_1stop", "n_1stop", "drop_0to1", "fare_2stop", "n_2stop", "drop_1to2"])

route,fare_0stop,n_0stop,fare_1stop,n_1stop,drop_0to1,fare_2stop,n_2stop,drop_1to2
str,f64,u32,f64,u32,f64,f64,u32,f64
"""LAX-BOS""",479.050915,19531,402.592439,61705,-76.46,431.406304,1553,28.81
"""BOS-LAX""",439.773329,20165,379.099464,57562,-60.67,472.597851,1047,93.5
"""ATL-LAX""",406.512017,18062,356.712119,67868,-49.8,413.613856,1058,56.9
"""LAX-ATL""",433.153445,17973,393.723521,61687,-39.43,396.051349,2143,2.33
"""JFK-LAX""",427.47898,39739,391.627357,33502,-35.85,437.260424,660,45.63
"""LAX-CLT""",460.503092,7585,444.562685,56808,-15.94,485.20895,3601,40.65
"""ORD-LGA""",177.332757,47830,162.498764,18680,-14.83,315.640374,187,153.14
"""LGA-ORD""",175.846446,47840,165.753704,19985,-10.09,260.880704,71,95.13
"""CLT-LAX""",432.921673,7399,422.976737,58672,-9.94,528.136113,3738,105.16
